# 7강 · Whisper로 한국어 음성 인식하기

위에서부터 순서대로 실행합니다. **1~4번이 핵심 실습**, 5~6번은 결과 활용입니다.

준비물: Python 3.9 이상인 주피터 환경, 한국어 음성 파일 한 개, 최초 설치·모델 다운로드용 인터넷.
기존 수업 파일은 이 노트북을 기준으로 `data/speech/1272-128104-0001.wav`에 둡니다. 다른 파일은 3번에서 경로만 바꿉니다.

[faster-whisper 공식 사용 안내](https://github.com/SYSTRAN/faster-whisper)


In [1]:
# 1. Whisper 실행 도구 설치 — 처음 한 번 실행합니다.
# faster-whisper는 사전학습된 Whisper 모델을 실행하는 라이브러리입니다.
# 이 수업은 API 키 없이 내 PC에서 실행합니다. 별도 FFmpeg 설치도 필요하지 않습니다.
# 설치 후 import 오류가 나면 주피터 커널을 재시작하고 2번부터 실행하세요.
# %pip install faster-whisper
# faster-whisper와 호환되는 av 버전을 함께 설치하므로 아래 한 줄만 실행합니다.
# 설치 후 커널을 재시작하고, 2번 ‘모델 불러오기’부터 순서대로 다시 실행하세요.
%pip install faster-whisper "av<19"

   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   --------- ------------------------------ 6.3/27.6 MB 35.1 MB/s eta 0:00:01
   ------------------- -------------------- 13.4/27.6 MB 32.3 MB/s eta 0:00:01
   ------------------------------- -------- 21.5/27.6 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------  27.5/27.6 MB 34.9 MB/s eta 0:00:01
   ---------------------------------------- 27.6/27.6 MB 32.4 MB/s  0:00:00
  Attempting uninstall: av
    Found existing installation: av 19.0.2
    Uninstalling av-19.0.2:
      Successfully uninstalled av-19.0.2
Note: you may need to restart the kernel to use updated packages.


In [2]:
# 2. 사전학습된 모델 불러오기
# 6강에서는 작은 데이터로 학습 구조를 살펴봤고, 이번에는 이미 학습된 모델을 사용합니다.
from faster_whisper import WhisperModel

# tiny: 빠른 실습용 다국어 모델입니다. 한국어에는 영어 전용 tiny.en을 쓰지 않습니다.
# cpu: GPU 없이 실행 / int8: 8비트 계산으로 메모리 부담을 줄입니다.
# 첫 실행에서는 models 폴더 아래로 모델을 다운로드한 뒤 메모리에 불러옵니다.
# 이후에는 저장된 모델 파일을 재사용합니다. 이 과정은 새로 학습하는 과정이 아닙니다.
model = WhisperModel("tiny", device="cpu", compute_type="int8", download_root="models")

# 경고 발생해도 문제 없느므로 진행
# HF_TOKEN 경고: 로그인 없이 다운로드한다는 안내이며, 다운로드가 진행되면 그대로 사용하면 됩니다.
# symlinks 경고: Windows에서 저장 공간을 더 사용할 수 있다는 안내이며, 음성 인식에는 문제가 없습니다.

In [3]:
# 3. 인식할 음성 파일 선택하고 듣기
from IPython.display import Audio, display

# 앞선 수업에서 사용한 한국어 WAV입니다. 경로는 현재 작업 폴더를 기준으로 적습니다.
# 내 녹음으로 바꾸려면 아래 값을 "my_voice.wav" 등 실제 파일 경로로 수정하세요.
audio_path = "data/speech/1272-128104-0001.wav"
display(Audio(filename=audio_path))

# 파일을 바꾼 뒤에는 이 셀과 4번 이후를 다시 실행합니다. 모델을 다시 불러올 필요는 없습니다.


In [4]:
# 4. 한국어 음성을 한국어 문장으로 변환하기 — 핵심 코드
# 파일을 전달하면 음성 읽기·16kHz 변환·Log-Mel 특징 추출을 내부에서 처리합니다.
# 5강의 lesson05_batch.npz나 정답 JSON을 별도로 넣지 않습니다.
segments, info = model.transcribe(
    audio_path,
    language="ko",       # 입력 음성이 한국어라고 지정합니다.
    task="transcribe",   # 들리는 언어 그대로 글로 옮깁니다. translate는 영어 번역입니다.
    beam_size=1,         # 후보 탐색 폭을 줄여 빠른 실습을 우선합니다.
)

# segments는 구간별 결과를 차례로 만드는 객체입니다.
# list로 끝까지 읽을 때 구간 인식이 진행되고, 결과를 다시 사용할 수 있게 보관합니다.
segments = list(segments)
text = "".join(segment.text for segment in segments).strip()
print(text)

# 출력은 모델의 예측이므로 실제 발음과 다를 수 있습니다.


안녕하세요. 오늘은 음성 데이터를 함께 살펴봅니다.


In [5]:
# 5. 인식 결과를 시간 구간과 함께 보기
# start·end는 초 단위의 예측 시각, text는 해당 구간의 인식 문장입니다.
# 단어 하나씩의 정확한 경계를 뜻하지는 않습니다.
for segment in segments:
    print(f"[{segment.start:.2f} ~ {segment.end:.2f}초] {segment.text.strip()}")


[0.00 ~ 1.28초] 안녕하세요.
[1.28 ~ 4.64초] 오늘은 음성 데이터를 함께 살펴봅니다.


In [6]:
# 6. 인식 문장을 텍스트 파일로 저장하기
from pathlib import Path

# 한글이 깨지지 않도록 UTF-8로 저장합니다. 다시 실행하면 같은 파일을 덮어씁니다.
output_path = Path(audio_path).with_name(Path(audio_path).stem + "_whisper.txt")
output_path.write_text(text, encoding="utf-8")
print("저장 위치:", output_path.resolve())

# 직접 해보기: 3번의 음성 경로를 다른 한국어 WAV로 바꾸고 3~6번을 실행하세요.


저장 위치: D:\OneDrive\AISource\2026_구름\음성\수업자료\7일차\data\speech\1272-128104-0001_whisper.txt


---
---

# 6강까지 배운 내용과 7강 Whisper 실습의 연결

## 1. 7강에는 왜 음성 전처리와 정답 전처리 코드가 없을까요?

7강은 **이미 학습된 Whisper 모델에 새로운 음성을 넣어 글자로 바꾸는 ‘추론’ 실습**입니다.

앞선 수업에서는 모델의 동작을 이해하기 위해 각 단계를 직접 작성했습니다.  
이번에는 사전학습 모델과 라이브러리가 제공하는 기능을 사용합니다.

| 구분 | 1~6강: 데이터 준비와 학습 과정 이해 | 7강: 사전학습 모델로 추론 |
|---|---|---|
| 음성 전처리 | 음성을 Log-Mel로 변환하고 입력 배열을 직접 준비 | 파일을 전달하면 라이브러리가 내부에서 처리 |
| 정답 전처리 | 정답 문장을 토큰 ID로 변환하여 학습에 사용 | 새로 학습하지 않으므로 필요 없음 |
| 모델 사용 | 모델 구조를 만들고 예측과 정답을 비교하며 학습 | 이미 학습된 가중치로 새로운 음성을 인식 |
| 결과 처리 | 모델의 숫자 출력을 해석 | 예측한 토큰을 글자로 변환한 결과를 사용 |

### 음성 전처리는 사라진 것이 아니라 내부에서 처리됩니다

```python
segments, info = model.transcribe(
    audio_path,
    language="ko",
    task="transcribe",
    beam_size=1,
)
```

위 코드에 **음성 파일 경로**를 전달하면 내부적으로 다음 과정이 이어집니다.

**음성 파일 읽기 → 모델에 맞는 파형 준비 → Log-Mel 특징 추출 → 모델 추론 → 인식 문장 반환**

따라서 5강에서 만든 Log-Mel 배열을 별도로 전달할 필요가 없습니다.  
전처리를 직접 구현할 때도 해당 모델이 요구하는 규격에 맞춰야 합니다.

### 정답 전처리는 학습할 때 필요합니다

학습에서는 모델의 예측이 얼마나 틀렸는지 계산해야 하므로 정답이 필요합니다.

- **학습:** 음성과 정답을 준비하고, 예측과 정답의 차이를 이용해 모델을 수정합니다.
- **추론:** 새로운 음성만 넣고, 이미 학습된 모델의 예측을 얻습니다.
- **평가:** 인식 결과를 실제 정답 문장과 비교해 성능을 측정합니다.

**7강에서는 추론을 하므로 정답을 모델에 전달하지 않습니다.**   
정답 문장은 이후 인식 성능을 평가할 때 사용할 수 있습니다.

## 2. 6강까지 배운 내용은 왜 중요할까요?

**모델을 실행하는 것과 모델을 올바르게 사용하는 것은 다릅니다.**

코드 몇 줄로 인식 결과를 얻을 수 있어도, 결과가 이상할 때 원인을 찾으려면 입력 데이터와 처리 과정을 이해해야 합니다.

| 배운 내용 | 이해해야 하는 이유 | 실제 프로젝트에서의 활용 |
|---|---|---|
| 파형·샘플링 레이트·모노 | 소리가 어떤 숫자로 저장되고 시간과 연결되는지 이해 | 녹음 파일 읽기, 구간 자르기, 채널·샘플링 레이트 변환 |
| FFT·STFT | 소리의 주파수 성분과 시간에 따른 변화를 이해 | 잡음과 음성의 특징 관찰, 스펙트로그램 분석 |
| Mel·Log-Mel | 모델이 소리를 어떤 특징으로 받아들이는지 이해 | 모델에 맞는 전처리 선택, 직접 학습할 입력 특징 준비 |
| 패딩·마스크·실제 길이 | 길이가 다른 음성을 함께 처리하는 원리 이해 | 여러 음성의 배치 처리, 패딩 구간 처리 |
| 음성과 정답 연결·토큰 ID | 입력 음성과 정답이 정확히 대응해야 하는 이유 이해 | 학습·평가 데이터 준비, 추가 학습 |
| 모델 구조·학습 과정 | 모델이 예측을 만들고 가중치를 수정하는 과정 이해 | 모델 선택, 학습 결과 해석, 성능 개선 방향 판단 |

### 결과가 잘못 나왔을 때 배운 내용이 도움이 됩니다

예를 들어 음성 인식 결과가 부정확하다면 다음을 구분해서 살펴볼 수 있습니다.

- **녹음 문제:** 목소리가 작거나 주변 소음이 큰가?
- **전처리 문제:** 음성을 자르면서 단어 일부를 잘랐는가?
- **입력 규격 문제:** 배열을 직접 전달하면서 샘플링 레이트를 잘못 적용했는가?
- **모델의 한계:** 전문용어나 특정 발음을 잘 인식하지 못하는가?

기초 원리를 알면 무조건 모델을 바꾸기 전에 **어느 단계에서 문제가 생겼는지 판단**할 수 있습니다.

## 3. 실제 프로젝트에서는 무엇을 직접 만들까요?

프로젝트 목적에 따라 직접 구현하는 범위가 달라집니다.

| 프로젝트 목적 | 주로 사용하는 방법 | 직접 만드는 부분 |
|---|---|---|
| 회의록·강의 자막 만들기 | Whisper 같은 사전학습 음성 인식 모델 활용 | 녹음·업로드, 모델 호출, 결과 표시·저장 |
| 시작·정지 등 몇 가지 음성 명령 구분 | Log-Mel 특징과 작은 분류 모델 활용 | 데이터 수집, 전처리, 정답 라벨, 모델 학습·평가 |
| 특정 분야의 인식 성능 개선 | 사전학습 모델의 추가 학습 검토 | 분야별 음성·정답 준비, 학습과 성능 비교 |

예를 들어 회의록 프로그램은 다음과 같이 구성할 수 있습니다.

**회의 녹음 → 음성 인식 모델로 글자 변환 → LLM으로 요약 → 회의록 표시·저장**

이때 음성 인식 모델을 처음부터 만들지는 않지만, 녹음 상태를 관리하고 인식 결과를 확인하며 서비스에 연결하는 작업은 직접 수행합니다.

## 6강까지 배운 기본 원이애 대한 코드를 활용해서 진행할 수 있는 프로젝트  
**6강까지 배운 내용을 직접 활용하려면, 음성을 분석하거나 몇 가지 소리·명령을 구분하는 프로젝트가 적합합니다.**     
자유로운 문장을 모두 받아쓰는 것보다 범위를 작게 정하면 직접 데이터 준비부터 학습까지 경험할 수 있습니다.

| 프로젝트 | 학생이 직접 할 일 | 활용하는 학습 내용 | 완성 결과 |
|---|---|---|---|
| **1. 녹음 품질 분석기** | 녹음 파일을 불러와 길이·최대 진폭·무음 비율을 구하고 파형과 스펙트로그램 표시 | 샘플링 레이트, 파형, STFT | 녹음 상태를 보여주는 분석 화면 |
| **2. 음성 앞뒤 무음 제거기** | 짧은 구간별 음량을 계산해 말소리의 시작과 끝을 찾고 잘라 저장 | 배열, 시간↔인덱스 변환, 음성 전처리 | 무음이 줄어든 WAV 파일 |
| **3. 소리 종류 분류기** | 박수·휘파람·말소리를 녹음하고 Log-Mel로 변환하여 분류 모델 학습 | Log-Mel, 정답 라벨, 패딩, 모델 학습 | 새 녹음을 넣으면 소리 종류 표시 |
| **4. 세 가지 음성 명령 분류기** | “시작·정지·대기”를 여러 번 녹음해 학습하고 새로운 녹음으로 예측 | 음성과 정답 연결, 특징 추출, 학습·평가 | 명령에 따라 화면의 상태 변경 |
| **5. 잡음에 따른 인식 변화 실험** | 같은 음성에 잡음의 크기를 다르게 섞고 특징과 분류 결과 비교 | 파형, STFT, Log-Mel, 모델 예측 | 잡음 크기별 결과 비교표 |

가장 추천하는 주제는 **‘시작·정지·대기’ 음성 명령 분류기**입니다. 6강까지 배운 내용을 하나의 흐름으로 연결할 수 있습니다.

1. **데이터 수집:** 여러 학생이 세 명령을 각각 반복해서 녹음합니다. 파일마다 명령과 화자 정보를 기록합니다.
2. **음성 전처리:** 샘플링 레이트와 채널을 맞추고, 앞뒤의 불필요한 무음을 줄입니다.
3. **특징 준비:** 음성을 Log-Mel로 변환하고, 길이가 다른 자료를 함께 입력할 수 있도록 준비합니다.
4. **정답 준비:** `시작=0, 정지=1, 대기=2`로 지정합니다. 이번에는 문장 전체가 아니라 **명령 종류 하나가 정답**입니다.
5. **모델 학습:** 세 종류 중 하나를 출력하는 작은 분류 모델을 학습합니다. 6강 모델의 출력과 손실함수도 분류 목적에 맞게 바꿉니다.
6. **새 음성 예측:** 학습에 사용하지 않은 녹음으로 시험하고, 예측 결과에 따라 화면에 “작동 중·정지·대기”를 표시합니다.

**학습한 녹음을 다시 맞히는 것과 새로운 사람의 목소리를 맞히는 것은 다릅니다.** 새 화자에게도 동작하는지 확인하려면 학습용과 평가용에 같은 사람이 섞이지 않도록 나눕니다.

이 프로젝트의 목표는 **세 명령을 구분하는 작은 모델을 직접 만드는 것**입니다.  
이후 “안녕하세요”나 주변 소리까지 입력해 보면, 세 명령 외의 소리를 처리하기 위한 **‘기타’ 데이터가 왜 필요한지**도 자연스럽게 이해할 수 있습니다.

